In [4]:
"""
Curate Project — Dataset Cleaner (with Guardrail Preservation)
===============================================================
Cleans curate_dataset_updated.csv only.

Instead of removing ALL refusal rows (which breaks the AI Docent's ability
to refuse unrelated questions), we keep a controlled ~2% sample so the
model learns BOTH behaviors:
  ✅ Answer artwork-related questions correctly
  ✅ Refuse unrelated/out-of-scope questions

Output:
  - curate_dataset_updated_clean.csv  ← use this for fine-tuning

Usage:
    python clean_updated_dataset.py

    For Colab: run each section as a cell, adjusting file paths as needed.
"""

import pandas as pd
import re
import os

# ── CONFIG ─────────────────────────────────────────────────────────────────────
UPDATED_CSV    = "curate_dataset_updated.csv"       # adjust path if needed
OUTPUT_UPDATED = "curate_dataset_updated_clean.csv"

REFUSAL_ANSWER = (
    "I am an AI Docent dedicated to this gallery's collection. "
    "I can only provide information and insights regarding the specific "
    "artwork and context we are currently viewing."
)

# ~2% refusal rate: model learns to refuse without collapsing to it
REFUSAL_TARGET_PCT = 0.02

RANDOM_SEED = 42  # for reproducibility
# ───────────────────────────────────────────────────────────────────────────────


def normalize_context(text):
    # Normalize Dalí accent variants throughout the full context string
    # (curate_dataset_updated.csv has mixed 'Dali' and 'Dalí' spellings)
    text = re.sub(r'\bDali\b', 'Dalí', text)
    return text.strip()


def clean_df(df, label, refusal_sample):
    initial = len(df)

    # Step 1: Strip whitespace and normalize context spelling
    for col in ["context", "question", "answer"]:
        df[col] = df[col].astype(str).str.strip()
    df["context"] = df["context"].apply(normalize_context)

    # Step 2: Separate refusal rows from normal rows
    refusal_mask = df["answer"] == REFUSAL_ANSWER
    df_refusals  = df[refusal_mask].copy()
    df_normal    = df[~refusal_mask].copy()
    print(f"  [{label}] Refusal rows found:       {len(df_refusals)} ({len(df_refusals)/initial*100:.1f}%)")

    # Step 3: Drop nulls and empty strings from normal rows
    df_normal = df_normal.dropna(subset=["context", "question", "answer"])
    df_normal = df_normal[df_normal["answer"].str.strip() != ""]
    df_normal = df_normal[df_normal["question"].str.strip() != ""]

    # Step 4: Deduplicate normal rows
    # First on (context + question + answer) — exact duplicates
    df_normal = df_normal.drop_duplicates(subset=["context", "question", "answer"], keep="first")
    # Then on (context + question) — same Q for same artwork, different answer
    df_normal = df_normal.drop_duplicates(subset=["context", "question"], keep="first")
    print(f"  [{label}] Normal rows (after dedup): {len(df_normal)}")

    # Step 5: Sample a controlled number of refusal rows (~2% of normal rows)
    actual_sample  = min(refusal_sample, len(df_refusals))
    df_ref_sampled = df_refusals.sample(n=actual_sample, random_state=RANDOM_SEED)
    print(f"  [{label}] Refusal rows kept:         {actual_sample} (~{actual_sample/(len(df_normal)+actual_sample)*100:.1f}% of dataset)")

    # Step 6: Recombine and shuffle
    df_out = (
        pd.concat([df_normal, df_ref_sampled], ignore_index=True)
        .sample(frac=1, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    print(f"  [{label}] Final row count:           {len(df_out)}\n")
    return df_out


def print_stats(df, label):
    refusal_count = (df["answer"] == REFUSAL_ANSWER).sum()
    top_count     = df["answer"].value_counts().iloc[0]
    top_pct       = top_count / len(df) * 100
    print(f"  {label}")
    print(f"    Rows:                      {len(df)}")
    print(f"    Unique artworks:           {df['context'].nunique()}")
    print(f"    Unique questions:          {df['question'].nunique()}")
    print(f"    Unique answers:            {df['answer'].nunique()}")
    print(f"    Refusal rows:              {refusal_count} ({refusal_count/len(df)*100:.1f}%)")
    print(f"    (context+question) dupes:  {df.duplicated(subset=['context','question']).sum()}")
    print(f"    Top answer dominance:      {top_count} ({top_pct:.1f}%)")
    print(f"    Avg answer length:         {df['answer'].str.split().str.len().mean():.1f} words")


# ── MAIN ───────────────────────────────────────────────────────────────────────

print("=" * 58)
print("STEP 1 — Load dataset")
print("=" * 58)
df_updated = pd.read_csv(UPDATED_CSV)
print(f"  Loaded: {UPDATED_CSV} ({len(df_updated)} rows)\n")

print("=" * 58)
print("STEP 2 — Calculate target refusal sample size")
print("=" * 58)
non_refusal_count = (df_updated["answer"] != REFUSAL_ANSWER).sum()
refusal_sample    = int(non_refusal_count * REFUSAL_TARGET_PCT)
print(f"  Non-refusal rows:           {non_refusal_count}")
print(f"  Target refusal sample (2%): {refusal_sample}\n")

print("=" * 58)
print("STEP 3 — Clean dataset")
print("=" * 58)
df_clean = clean_df(df_updated.copy(), "updated", refusal_sample)

print("=" * 58)
print("STEP 4 — Save cleaned dataset")
print("=" * 58)
df_clean.to_csv(OUTPUT_UPDATED, index=False)
print(f"  Saved: {OUTPUT_UPDATED}")
print(f"  Size:  {os.path.getsize(OUTPUT_UPDATED) / 1024:.1f} KB\n")

print("=" * 58)
print("FINAL STATS")
print("=" * 58)
print_stats(df_clean, OUTPUT_UPDATED)

print("\n✅ Done! Use curate_dataset_updated_clean.csv for fine-tuning.")
print("   The model will learn to answer artwork questions AND refuse unrelated ones.")

STEP 1 — Load dataset
  Loaded: curate_dataset_updated.csv (2936 rows)

STEP 2 — Calculate target refusal sample size
  Non-refusal rows:           2816
  Target refusal sample (2%): 56

STEP 3 — Clean dataset
  [updated] Refusal rows found:       120 (4.1%)
  [updated] Normal rows (after dedup): 2814
  [updated] Refusal rows kept:         56 (~2.0% of dataset)
  [updated] Final row count:           2870

STEP 4 — Save cleaned dataset
  Saved: curate_dataset_updated_clean.csv
  Size:  4608.6 KB

FINAL STATS
  curate_dataset_updated_clean.csv
    Rows:                      2870
    Unique artworks:           14
    Unique questions:          2466
    Unique answers:            351
    Refusal rows:              56 (2.0%)
    (context+question) dupes:  0
    Top answer dominance:      56 (2.0%)
    Avg answer length:         11.3 words

✅ Done! Use curate_dataset_updated_clean.csv for fine-tuning.
   The model will learn to answer artwork questions AND refuse unrelated ones.
